[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abpandey-dev/learn-python-from-scratch-to-advanced/blob/main/notebooks/06-dsa-interview/06_linked-list.ipynb)

*DSA Interview Handbook · Notebook 6 of 14*

# Linked List

Nodes joined by pointers — master dummy heads, fast/slow pointers and in-place reversal.

### 👩‍🏫 From your teacher

**The picture to keep in your head:** A linked list is a treasure hunt: each clue only tells you where the next one is.

**What interviewers really test here:** Dummy nodes, fast and slow pointers, and reversing without losing the rest.

### How to use this notebook

1. **Run the setup cell** just below.
2. **Read each lesson slowly.** Work the numbers on paper before reading on.
3. **Run and then change the examples.** Breaking code on purpose is how you learn.
4. **Try each practice question before opening an answer.** If you're stuck, open one hint at a time.
5. **Say your interview answers out loud.** Knowing something and explaining it are different skills.

### Your roadmap

- [ ] 1. Nodes and pointers *(Basics)*
- [ ] 2. Insert, delete and the dummy head *(Basics)*
- [ ] 3. Reversing a list *(Core)*
- [ ] 4. Fast and slow pointers *(Core)*
- [ ] 5. Merging and splitting *(Core)*
- [ ] 6. Combining techniques *(Advanced)*
- [ ] 20 practice questions
- [ ] Interview questions, out loud

In [1]:
# ▶ Run this cell first. It prepares the notebook; you don't need to read it.
import contextlib, copy, io, math, sys
import numpy as np, matplotlib.pyplot as plt
%matplotlib inline

PLAIN = False

class ListNode:
    def __init__(self, val=0, next=None):
        self.val, self.next = val, next

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def mk_list(vals, pos=-1):
    dummy = cur = ListNode()
    nodes = []
    for v in vals:
        cur.next = ListNode(v)
        cur = cur.next
        nodes.append(cur)
    if pos >= 0 and nodes:
        nodes[-1].next = nodes[pos]
    return dummy.next

def list_arr(head):
    out, seen = [], set()
    while head and id(head) not in seen:
        seen.add(id(head))
        out.append(head.val)
        head = head.next
    return out

def mk_tree(vals):
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    queue, i = [root], 1
    while queue and i < len(vals):
        node = queue.pop(0)
        for side in ("left", "right"):
            if i < len(vals) and vals[i] is not None:
                child = TreeNode(vals[i])
                setattr(node, side, child)
                queue.append(child)
            i += 1
    return root

def tree_arr(root):
    out, queue = [], [root]
    while queue:
        node = queue.pop(0)
        if node:
            out.append(node.val)
            queue += [node.left, node.right]
        else:
            out.append(None)
    while out and out[-1] is None:
        out.pop()
    return out

def find(root, val):
    if not root or root.val == val:
        return root
    return find(root.left, val) or find(root.right, val)

ListNode.__repr__ = lambda self: f"ListNode{list_arr(self)}"
TreeNode.__repr__ = lambda self: f"TreeNode{tree_arr(self)}"

def plain(x, digits=4):
    """NumPy / pandas results as plain Python values, so they compare with a test's expected answer."""
    np = sys.modules.get("numpy")
    pd = sys.modules.get("pandas")
    if pd is not None:
        if isinstance(x, pd.DataFrame):
            return {str(c): plain(x[c].tolist(), digits) for c in x.columns}
        if isinstance(x, pd.Series):
            return {plain(k, digits): plain(v, digits) for k, v in x.items()}
        if isinstance(x, pd.Index):
            return plain(x.tolist(), digits)
        if x is pd.NaT:
            return None
        if isinstance(x, pd.Timestamp):
            return str(x.date()) if x == x.normalize() else str(x)
    if np is not None:
        if isinstance(x, np.ndarray):
            return plain(x.tolist(), digits)
        if isinstance(x, np.generic):
            return plain(x.item(), digits)
    if isinstance(x, float):
        return None if x != x else round(x, digits) + 0.0
    if isinstance(x, list):
        return [plain(v, digits) for v in x]
    if isinstance(x, tuple):
        return tuple(plain(v, digits) for v in x)
    if isinstance(x, dict):
        return {plain(k, digits): plain(v, digits) for k, v in x.items()}
    if isinstance(x, (set, frozenset)):
        return {plain(v, digits) for v in x}
    return x


def check(fn, tests, plain_compare=None, listify=False, design=False, prep=None, post=None,
          norm=None, verify=None, runner=None, inplace=None):
    """Run your code against the tests and show which pass."""
    use_plain = PLAIN if plain_compare is None else plain_compare
    passed, problems = 0, []
    for args, want in tests:
        try:
            a = copy.deepcopy(args)
            if design:
                obj = fn(*a[0])
                got = [getattr(obj, m)(*p) for m, p in a[1]]
            elif runner is not None:
                got = runner(fn, a)
            else:
                if prep is not None:
                    a = prep(a)
                with contextlib.redirect_stdout(io.StringIO()):
                    got = fn(*a)
                if inplace is not None:
                    got = a[inplace]
                if listify:
                    got = list(got)
                if post is not None:
                    got = post(got)
            if verify is not None:
                ok = bool(verify(copy.deepcopy(args), got))
            elif norm is not None:
                ok = norm(copy.deepcopy(got)) == norm(copy.deepcopy(want))
            else:
                ok = (plain(got), plain(want)) if use_plain else (got, want)
                ok = ok[0] == ok[1]
            ok = ok if isinstance(ok, bool) else False
        except Exception as e:  # your code raised an error: report it, don't stop the notebook
            ok, got = False, f"{type(e).__name__}: {e}"
        passed += ok
        if not ok:
            problems.append((args, want, got))
    plt.close("all")
    if passed == len(tests):
        print(f"✅ All {len(tests)} tests passed. Well done! Now compare with the answers below.")
    else:
        print(f"❌ {passed}/{len(tests)} tests passed. First one to fix:")
        args, want, got = problems[0]
        print("   input:   ", repr(args)[:300])
        print("   expected:", repr(want)[:300])
        print("   you got: ", repr(got)[:300])
    return None

print("Ready! Scroll down and start learning.")

Ready! Scroll down and start learning.


---
# Part 1: The theory in one place

Read this once for the big picture. It's fine if it doesn't all stick yet: the lessons will make it concrete, and you can come back to revise.

### Definition

A **linked list** is a linear data structure in which elements, called **nodes**, are not stored contiguously. Each node contains **data** and a **reference (pointer)** to the next node. The first node is the **head**; the last node points to `None` (null).



```text
head → [10 | •] → [20 | •] → [30 | None]
```



The list is accessed only through the head, and elements are reached by following pointers one by one (**sequential access**).

### Types of linked lists

| Type | Each node has | Picture | Use |
| --- | --- | --- | --- |
| **Singly linked** | data, next | `A → B → C → None` | stacks, simple lists, most interview problems |
| **Doubly linked** | data, prev, next | `None ⇄ A ⇄ B ⇄ C ⇄ None` | LRU cache, browser history, deque |
| **Circular singly** | data, next; last → head | `A → B → C → A` | round-robin scheduling, circular buffers |
| **Circular doubly** | prev, next; ends connected | ring both ways | Fibonacci heaps, music playlists on repeat |



A **sentinel** (dummy) node is an extra node that never holds real data, placed at the head (and/or tail) to remove special cases for empty lists and head changes.

### Operations and their complexity

| Operation | Singly | Doubly | How |
| --- | --- | --- | --- |
| Access k-th element | O(k) | O(k) | walk from head |
| Search for a value | O(n) | O(n) | walk and compare |
| Insert at head | O(1) | O(1) | `new.next = head; head = new` |
| Insert at tail | O(n), O(1) with tail pointer | O(1) with tail | link from the tail |
| Insert after a given node | O(1) | O(1) | `x.next = p.next; p.next = x` |
| Delete head | O(1) | O(1) | `head = head.next` |
| Delete a given node | O(n) (need the previous node) | **O(1)** | doubly: `node.prev.next = node.next` |
| Reverse | O(n) | O(n) | flip pointers in one pass |



Space: O(n) for the nodes, plus one pointer per node (two for doubly linked).

### Linked list vs array

|   | Array | Linked list |
| --- | --- | --- |
| Memory layout | contiguous | scattered nodes |
| Access by index | O(1) | O(n) |
| Insert/delete at front | O(n) | O(1) |
| Insert/delete in middle (node known) | O(n) | O(1) |
| Size | fixed or amortised resize | grows one node at a time |
| Extra memory | spare capacity | pointer(s) per element |
| Cache performance | excellent | poor (pointer chasing) |
| Binary search | yes | not efficiently |

### Advantages, disadvantages and applications

**👍 Pros**

##### Advantages

- Dynamic size, no reallocation
- O(1) insert/delete at the head or next to a known node
- No large contiguous block of memory needed
- Easy to splice, split and merge lists



**👎 Cons**

##### Disadvantages

- No random access; O(n) to reach an element
- Extra memory for pointers
- Poor cache locality → slower traversal than arrays
- Pointer bugs: lost nodes, accidental cycles





#### Applications




- Implementing stacks, queues and deques.
- Hash table chaining (each bucket is a list).
- LRU caches (hash map + doubly linked list).
- Adjacency lists for graphs.
- Undo history, music playlists, image viewers (next/previous).
- Memory allocators' free lists; polynomial arithmetic.

### Key techniques (summary)

| Technique | Solves |
| --- | --- |
| Dummy head | any problem where the head may change |
| Fast & slow pointers | middle node, cycle detection, cycle start, palindrome check |
| Gap pointers | n-th node from the end in one pass |
| In-place reversal | reverse list / sub-list / k-groups |
| Split + merge | merge sort on lists, reorder list |
| Hash map of nodes | copying lists with random pointers, intersection (O(n) space) |

---
# Part 2: Lessons, step by step

## Lesson 1: Nodes and pointers

*Level: Basics*

An array keeps items in neighbouring boxes. A **linked list** keeps each item in its own **node**, anywhere in memory, and each node stores a **pointer** (reference) to the next node.



```text
head → [3 | •] → [7 | •] → [1 | •] → None
        val  next
```



```python
class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next
```



You only hold the `head`. To reach the k-th node you must walk k steps — there is no `list[k]`. In return, once you hold a node, inserting or removing next to it is O(1): just change arrows, nothing shifts.



```python
cur = head
while cur:              # the standard walk
    print(cur.val)
    cur = cur.next
```

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [2]:
def build_and_walk():
    vals = [3, 7, 1, 9]
    head = None
    for v in reversed(vals):
        head = ListNode(v, head)
    cur, total = head, 0
    while cur:
        total += cur.val
        cur = cur.next
    return total


build_and_walk()

20

<details>
<summary><b>🔎 What happens, step by step</b></summary>

1. Create node 9 pointing to the old head
2. Create node 1 pointing to the old head
3. Create node 7 pointing to the old head
4. Create node 3 pointing to the old head
5. Walk: visit 3, running total 3
6. Walk: visit 7, running total 10
7. Walk: visit 1, running total 11
8. Walk: visit 9, running total 20

</details>


**✏️ Your turn:** you now have what you need for [Q7](#q7). Try one now, or carry on and do them in Part 3.

## Lesson 2: Insert, delete and the dummy head

*Level: Basics*

To **insert** node `x` after node `p`: `x.next = p.next; p.next = x` (order matters — save the rest first).



To **delete** the node after `p`: `p.next = p.next.next`.



Both need the node *before* the target. But the head has no node before it, which forces special cases. The fix is a **dummy** node placed in front:



```python
dummy = ListNode(0, head)
prev = dummy
while prev.next:
    if prev.next.val == target:
        prev.next = prev.next.next   # works even if head is deleted
    else:
        prev = prev.next
return dummy.next                    # the real (maybe new) head
```



**💡 Tip:** Whenever the head might change — merge, delete, partition, swap — start with a dummy.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [3]:
def dummy_delete():
    head = mk_list([6, 2, 6, 3, 6])
    dummy = ListNode(0, head)
    prev = dummy
    while prev.next:
        if prev.next.val == 6:
            prev.next = prev.next.next
        else:
            prev = prev.next
    return dummy.next


dummy_delete()

ListNode[2, 3]

<details>
<summary><b>🔎 What happens, step by step</b></summary>

1. prev.next is 6 → skip it (prev.next = prev.next.next)
2. Keep this node, step forward
3. prev.next is 6 → skip it (prev.next = prev.next.next)
4. Keep this node, step forward
5. prev.next is 6 → skip it (prev.next = prev.next.next)

</details>


**✏️ Your turn:** you now have what you need for [Q9](#q9), [Q18](#q18), [Q16](#q16). Try one now, or carry on and do them in Part 3.

## Lesson 3: Reversing a list

*Level: Core*

Reversal is the most common linked-list operation. Walk the list and point each node **backwards**. You need three names:




- `prev` — the already-reversed part (starts as `None`),
- `cur` — the node being flipped,
- `nxt` — saved before flipping, or the rest of the list is lost.



```python
prev, cur = None, head
while cur:
    nxt = cur.next
    cur.next = prev
    prev, cur = cur, nxt
return prev
```



Reversing a **part** of the list (positions left..right, or groups of k) is the same loop plus reconnecting the reversed piece to what comes before and after it.

**✏️ Your turn:** you now have what you need for [Q1](#q1), [Q17](#q17), [Q20](#q20). Try one now, or carry on and do them in Part 3.

## Lesson 4: Fast and slow pointers

*Level: Core*

Move `slow` one step and `fast` two steps at a time.




- **Middle**: when fast reaches the end, slow is in the middle.
- **Cycle**: if the list loops, fast eventually laps slow and they meet. If fast reaches `None`, there's no cycle. O(1) memory, unlike a visited set.
- **Cycle start**: after they meet, put one pointer back at head and move both one step at a time — they meet at the cycle's entrance (follows from the distances walked).
- **n-th from the end**: give one pointer an n-step head start, then move both; when the leader hits the end, the follower is n from the end.



**⚠️ Watch out:** Always check `while fast and fast.next` before `fast.next.next`.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [4]:
def cycle_lesson():
    head = mk_list([1, 2, 3, 4, 5, 6], 2)
    slow = fast = head
    while fast and fast.next:
        slow, fast = slow.next, fast.next.next
        if slow is fast:
            return True
    return False


cycle_lesson()

True

<details>
<summary><b>🔎 What happens, step by step</b></summary>

1. slow moves 1, fast moves 2
2. slow moves 1, fast moves 2
3. slow moves 1, fast moves 2
4. slow moves 1, fast moves 2
5. They met → the list has a cycle

</details>


**✏️ Your turn:** you now have what you need for [Q4](#q4), [Q3](#q3), [Q8](#q8), [Q9](#q9). Try one now, or carry on and do them in Part 3.

## Lesson 5: Merging and splitting

*Level: Core*

- **Merge two sorted lists**: a dummy and a `tail`; attach the smaller head each step, then attach whatever remains.
- **Partition**: build two separate chains (less / rest) with two dummies, then join them. Cut the last tail's `next` to avoid a cycle.
- **Merge k lists**: a min-heap of the current heads, O(N log k).
- **Sort a list**: merge sort — split at the middle with fast/slow, sort both halves, merge. O(n log n) with no random access needed.

**✏️ Your turn:** you now have what you need for [Q2](#q2), [Q18](#q18), [Q19](#q19), [Q15](#q15), [Q14](#q14), [Q11](#q11). Try one now, or carry on and do them in Part 3.

## Lesson 6: Combining techniques

*Level: Advanced*

Harder problems chain the basic moves:




- **Reorder list** (L0 → Ln → L1 → Ln−1 …) = find middle + reverse second half + weave the halves.
- **Palindrome list** = find middle + reverse second half + compare.
- **Rotate by k** = find length and tail, link tail to head (ring), cut at the right place.
- **Intersection of two lists** = pointer a walks A then B, pointer b walks B then A; both travel the same distance and meet at the junction.



#### LRU cache — the classic design question



You need O(1) lookup *and* O(1) "move to most recent" *and* O(1) "remove least recent". A dict gives the lookup; a **doubly linked list** (each node has `prev` and `next`) gives O(1) removal from anywhere. The dict maps key → node; the list is ordered by recency, with sentinel head and tail nodes so there are no edge cases.

**✏️ Your turn:** you now have what you need for [Q10](#q10), [Q5](#q5), [Q13](#q13), [Q6](#q6), [Q12](#q12). Try one now, or carry on and do them in Part 3.

---
# Part 3: Practice like an interview

There are 20 questions here, easiest first.

**Work the way you would in a real interview:** restate the problem, try an example by hand, say the simple approach and its cost, improve it, then write the code. Test it with the cell. Only open an answer once you've made a real attempt, because struggling is where the learning happens.

**▶ Run this cell before the questions.** It sets up the data and tools they share.

In [5]:
L1 = lambda a: (mk_list(a[0]),)


<a id="q1"></a>
### Q1. Reverse Linked List

🟢 Warm-up · *In-place reversal*

Reverse a singly linked list and return the new head.

**Examples**

- `head = [1, 2, 3, 4, 5]` → `[5, 4, 3, 2, 1]`
- `head = [1, 2]` → `[2, 1]`
- `head = []` → `[]`

In [ ]:
def reverse_list(head):
    """
    Reverse a singly linked list and return the new head.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(reverse_list, [(([1, 2, 3, 4, 5],), [5, 4, 3, 2, 1]), (([1, 2],), [2, 1]), (([],), [])], prep=L1, post=list_arr)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Copy the values into an array and build a new list from the back.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Walk the list once with prev / cur and point each node back at prev.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Copy the values into an array and build a new list from the back.

```python
def reverse_list(head):
    vals = []
    while head:
        vals.append(head.val)
        head = head.next
    dummy = cur = ListNode()
    for v in reversed(vals):
        cur.next = ListNode(v)
        cur = cur.next
    return dummy.next
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Walk the list once with prev / cur and point each node back at prev.

```python
def reverse_list(head):
    prev, cur = None, head
    while cur:
        nxt = cur.next
        cur.next = prev
        prev, cur = cur, nxt
    return prev
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Copy the values into an array and build a new list from the back. That costs O(n) time and O(n) space."

"I can do better: Walk the list once with prev / cur and point each node back at prev. That's O(n) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `head = [1, 2, 3, 4, 5]`

1. Node 1 now points back; move on
2. Node 2 now points back; move on
3. Node 3 now points back; move on
4. Node 4 now points back; move on
5. Node 5 now points back; move on

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/linked-list/reverse-linked-list)

<a id="q2"></a>
### Q2. Merge Two Sorted Lists

🟢 Warm-up · *Dummy head · splice*

Merge two sorted linked lists into one sorted list by splicing their nodes.

**Examples**

- `l1 = [1, 2, 4], l2 = [1, 3, 4]` → `[1, 1, 2, 3, 4, 4]`
- `l1 = [], l2 = []` → `[]`
- `l1 = [], l2 = [0]` → `[0]`

In [ ]:
def merge_two_lists(l1, l2):
    """
    Merge two sorted linked lists into one sorted list by splicing their nodes.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(merge_two_lists, [(([1, 2, 4], [1, 3, 4]), [1, 1, 2, 3, 4, 4]), (([], []), []), (([], [0]), [0])], prep=lambda a: (mk_list(a[0]), mk_list(a[1])), post=list_arr)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Collect all values, sort them, build a new list.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A dummy node and a tail pointer. Attach the smaller head each step; when one list runs out, attach the rest of the other.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Collect all values, sort them, build a new list.

```python
def merge_two_lists(l1, l2):
    vals = []
    for node in (l1, l2):
        while node:
            vals.append(node.val)
            node = node.next
    dummy = cur = ListNode()
    for v in sorted(vals):
        cur.next = ListNode(v)
        cur = cur.next
    return dummy.next
```

**Cost:** O((m+n) log(m+n)) time, O(m + n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** A dummy node and a tail pointer. Attach the smaller head each step; when one list runs out, attach the rest of the other.

```python
def merge_two_lists(l1, l2):
    dummy = tail = ListNode()
    while l1 and l2:
        if l1.val <= l2.val:
            tail.next, l1 = l1, l1.next
        else:
            tail.next, l2 = l2, l2.next
        tail = tail.next
    tail.next = l1 or l2
    return dummy.next
```

**Cost:** O(m + n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Collect all values, sort them, build a new list. That costs O((m+n) log(m+n)) time and O(m + n) space."

"I can do better: A dummy node and a tail pointer. Attach the smaller head each step; when one list runs out, attach the rest of the other. That's O(m + n) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `l1 = [1, 2, 4], l2 = [1, 3, 4]`

1. Attach 1
2. Attach 1
3. Attach 2
4. Attach 3
5. Attach 4

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/linked-list/merge-two-sorted-lists)

<a id="q3"></a>
### Q3. Linked List Cycle

🟢 Warm-up · *Fast & slow*

Return `True` if the list has a cycle (some node's next points back to an earlier node).

**Examples**

- `head = [3, 2, 0, -4], pos = 1` → `True`
- `head = [1, 2], pos = 0` → `True`
- `head = [1], pos = -1` → `False`

In [ ]:
def has_cycle(head):
    """
    Return True if the list has a cycle (some node's next points back to an
    earlier node).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(has_cycle, [(([3, 2, 0, -4], 1), True), (([1, 2], 0), True), (([1], -1), False)], prep=lambda a: (mk_list(a[0], a[1]),))

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Store every visited node in a set; a repeat means a cycle.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Floyd: slow moves 1, fast moves 2. If there's a cycle fast laps slow and they meet; otherwise fast hits None.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Store every visited node in a set; a repeat means a cycle.

```python
def has_cycle(head):
    seen = set()
    while head:
        if head in seen:
            return True
        seen.add(head)
        head = head.next
    return False
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Floyd: slow moves 1, fast moves 2. If there's a cycle fast laps slow and they meet; otherwise fast hits None.

```python
def has_cycle(head):
    slow = fast = head
    while fast and fast.next:
        slow, fast = slow.next, fast.next.next
        if slow is fast:
            return True
    return False
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Store every visited node in a set; a repeat means a cycle. That costs O(n) time and O(n) space."

"I can do better: Floyd: slow moves 1, fast moves 2. If there's a cycle fast laps slow and they meet; otherwise fast hits None. That's O(n) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `head = [3, 2, 0, -4], pos = 1`

1. slow +1, fast +2
2. slow +1, fast +2
3. slow +1, fast +2
4. They met → cycle

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/linked-list/linked-list-cycle)

<a id="q4"></a>
### Q4. Middle of the Linked List

🟢 Warm-up · *Fast & slow*

Return the middle node (the second middle if there are two). Shown as the list from that node.

**Examples**

- `head = [1, 2, 3, 4, 5]` → `[3, 4, 5]`
- `head = [1, 2, 3, 4, 5, 6]` → `[4, 5, 6]`

In [ ]:
def middle_node(head):
    """
    Return the middle node (the second middle if there are two). Shown as the
    list from that node.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(middle_node, [(([1, 2, 3, 4, 5],), [3, 4, 5]), (([1, 2, 3, 4, 5, 6],), [4, 5, 6])], prep=L1, post=list_arr)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Count the length, then walk length // 2 steps.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

When fast (2 steps) reaches the end, slow (1 step) is in the middle.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Count the length, then walk length // 2 steps.

```python
def middle_node(head):
    n, cur = 0, head
    while cur:
        n, cur = n + 1, cur.next
    for _ in range(n // 2):
        head = head.next
    return head
```

**Cost:** O(n) — two passes time, O(1) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** When fast (2 steps) reaches the end, slow (1 step) is in the middle.

```python
def middle_node(head):
    slow = fast = head
    while fast and fast.next:
        slow, fast = slow.next, fast.next.next
    return slow
```

**Cost:** O(n) — one pass time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Count the length, then walk length // 2 steps. That costs O(n) — two passes time and O(1) space."

"I can do better: When fast (2 steps) reaches the end, slow (1 step) is in the middle. That's O(n) — one pass time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `head = [1, 2, 3, 4, 5]`

1. Both start at head
2. slow +1, fast +2
3. slow +1, fast +2

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/linked-list/middle-of-the-linked-list)

<a id="q5"></a>
### Q5. Palindrome Linked List

🟢 Warm-up · *Reverse the second half*

Return `True` if the list's values read the same forwards and backwards.

**Examples**

- `head = [1, 2, 2, 1]` → `True`
- `head = [1, 2]` → `False`
- `head = [1, 2, 3, 2, 1]` → `True`

In [ ]:
def is_palindrome_list(head):
    """
    Return True if the list's values read the same forwards and backwards.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(is_palindrome_list, [(([1, 2, 2, 1],), True), (([1, 2],), False), (([1, 2, 3, 2, 1],), True)], prep=L1)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Copy values to an array and compare it with its reverse.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Find the middle, reverse the second half, compare the halves node by node.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Copy values to an array and compare it with its reverse.

```python
def is_palindrome_list(head):
    vals = []
    while head:
        vals.append(head.val)
        head = head.next
    return vals == vals[::-1]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Find the middle, reverse the second half, compare the halves node by node.

```python
def is_palindrome_list(head):
    slow = fast = head
    while fast and fast.next:
        slow, fast = slow.next, fast.next.next
    prev = None
    while slow:
        slow.next, prev, slow = prev, slow, slow.next
    a, b = head, prev
    while b:
        if a.val != b.val:
            return False
        a, b = a.next, b.next
    return True
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Copy values to an array and compare it with its reverse. That costs O(n) time and O(n) space."

"I can do better: Find the middle, reverse the second half, compare the halves node by node. That's O(n) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `head = [1, 2, 3, 3, 2, 1]`

1. Compare 1 and 1
2. Compare 2 and 2
3. Compare 3 and 3

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/linked-list/palindrome-linked-list)

<a id="q6"></a>
### Q6. Intersection of Two Linked Lists

🟢 Warm-up · *Switch heads*

Two lists may merge into a shared tail. Return the first shared node (shown by value), or `None`.

**Examples**

- `onlyA = [4, 1], onlyB = [5, 6, 1], shared = [8, 4, 5]` → `8`
- `onlyA = [1, 9, 1], onlyB = [3], shared = [2, 4]` → `2`
- `onlyA = [2, 6, 4], onlyB = [1, 5], shared = []` → `None`

In [ ]:
def _shared(a, b, common):
    tail = mk_list(common)

    def build(vals):
        if not vals:
            return tail
        head = mk_list(vals)
        cur = head
        while cur.next:
            cur = cur.next
        cur.next = tail
        return head
    return build(a), build(b)

def get_intersection_node(head_a, head_b):
    """
    Two lists may merge into a shared tail. Return the first shared node (shown
    by value), or None.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(get_intersection_node, [(([4, 1], [5, 6, 1], [8, 4, 5]), 8), (([1, 9, 1], [3], [2, 4]), 2), (([2, 6, 4], [1, 5], []), None)], prep=lambda a: _shared(*a), post=lambda n: n.val if n else None)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Put every node of A in a set, then walk B until a node is in the set.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Pointer a walks A then B; pointer b walks B then A. Both travel m + n steps, so they line up at the intersection (or both reach None).

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Put every node of A in a set, then walk B until a node is in the set.

```python
def get_intersection_node(head_a, head_b):
    seen = set()
    while head_a:
        seen.add(head_a)
        head_a = head_a.next
    while head_b:
        if head_b in seen:
            return head_b
        head_b = head_b.next
    return None
```

**Cost:** O(m + n) time, O(m) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Pointer a walks A then B; pointer b walks B then A. Both travel m + n steps, so they line up at the intersection (or both reach None).

```python
def get_intersection_node(head_a, head_b):
    a, b = head_a, head_b
    while a is not b:
        a = a.next if a else head_b
        b = b.next if b else head_a
    return a
```

**Cost:** O(m + n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Put every node of A in a set, then walk B until a node is in the set. That costs O(m + n) time and O(m) space."

"I can do better: Pointer a walks A then B; pointer b walks B then A. Both travel m + n steps, so they line up at the intersection (or both reach None). That's O(m + n) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `onlyA = [4, 1], onlyB = [5, 6, 1], shared = [8, 4, 5]`

1. a at 4, b at 5
2. a at 1, b at 6
3. a at 8, b at 1
4. a at 4, b at 8
5. a at 5, b at 4
6. a at None, b at 5
7. a at 5, b at None
8. a at 6, b at 4
9. a at 1, b at 1
10. Same node: 8

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/linked-list/intersection-of-two-linked-lists)

<a id="q7"></a>
### Q7. Remove Duplicates from Sorted List

🟢 Warm-up · *Skip equal neighbours*

Delete duplicates from a sorted list so each value appears once.

**Examples**

- `head = [1, 1, 2]` → `[1, 2]`
- `head = [1, 1, 2, 3, 3]` → `[1, 2, 3]`

In [ ]:
def delete_duplicates(head):
    """
    Delete duplicates from a sorted list so each value appears once.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(delete_duplicates, [(([1, 1, 2],), [1, 2]), (([1, 1, 2, 3, 3],), [1, 2, 3])], prep=L1, post=list_arr)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Collect unique values and rebuild the list.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

While cur.next has the same value as cur, bypass it.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Collect unique values and rebuild the list.

```python
def delete_duplicates(head):
    vals = []
    while head:
        if not vals or vals[-1] != head.val:
            vals.append(head.val)
        head = head.next
    return mk_list(vals)
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** While cur.next has the same value as cur, bypass it.

```python
def delete_duplicates(head):
    cur = head
    while cur and cur.next:
        if cur.next.val == cur.val:
            cur.next = cur.next.next
        else:
            cur = cur.next
    return head
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Collect unique values and rebuild the list. That costs O(n) time and O(n) space."

"I can do better: While cur.next has the same value as cur, bypass it. That's O(n) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `head = [1, 1, 2, 3, 3, 3, 4]`

1. Next node repeats 1 → bypass it
2. Next node repeats 3 → bypass it
3. Next node repeats 3 → bypass it

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/linked-list/remove-duplicates-from-sorted-list)

<a id="q8"></a>
### Q8. Linked List Cycle II

🟡 Interview standard · *Floyd · find the entrance*

Return the node where the cycle begins, or `None`. (The answer shown is that node's value.)

**Examples**

- `head = [3, 2, 0, -4], pos = 1` → `2`
- `head = [1, 2], pos = 0` → `1`
- `head = [1], pos = -1` → `None`

In [ ]:
def detect_cycle(head):
    """
    Return the node where the cycle begins, or None. (The answer shown is that
    node's value.)
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(detect_cycle, [(([3, 2, 0, -4], 1), 2), (([1, 2], 0), 1), (([1], -1), None)], prep=lambda a: (mk_list(a[0], a[1]),), post=lambda n: n.val if n else None)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Walk with a set; the first node seen twice is the entrance.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

After slow and fast meet, restart one pointer at head and move both by 1. The distance math (a = k·c − b) makes them meet exactly at the cycle entrance.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Walk with a set; the first node seen twice is the entrance.

```python
def detect_cycle(head):
    seen = set()
    while head:
        if head in seen:
            return head
        seen.add(head)
        head = head.next
    return None
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** After slow and fast meet, restart one pointer at head and move both by 1. The distance math (a = k·c − b) makes them meet exactly at the cycle entrance.

```python
def detect_cycle(head):
    slow = fast = head
    while fast and fast.next:
        slow, fast = slow.next, fast.next.next
        if slow is fast:
            break
    else:
        return None
    p = head
    while p is not slow:
        p, slow = p.next, slow.next
    return p
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Walk with a set; the first node seen twice is the entrance. That costs O(n) time and O(n) space."

"I can do better: After slow and fast meet, restart one pointer at head and move both by 1. The distance math (a = k·c − b) makes them meet exactly at the cycle entrance. That's O(n) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `head = [1, 2, 3, 4, 5, 6], pos = 2`

1. Phase 1: slow +1, fast +2
2. Phase 1: slow +1, fast +2
3. Phase 1: slow +1, fast +2
4. Phase 1: slow +1, fast +2
5. Phase 2: from head and from the meeting point, both +1
6. Phase 2: from head and from the meeting point, both +1
7. They meet at 3: the cycle entrance

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/linked-list/linked-list-cycle-ii)

<a id="q9"></a>
### Q9. Remove Nth Node From End of List

🟡 Interview standard · *Gap pointers*

Remove the n-th node from the end and return the head.

**Examples**

- `head = [1, 2, 3, 4, 5], n = 2` → `[1, 2, 3, 5]`
- `head = [1], n = 1` → `[]`
- `head = [1, 2], n = 1` → `[1]`

In [ ]:
def remove_nth_from_end(head, n):
    """
    Remove the n-th node from the end and return the head.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(remove_nth_from_end, [(([1, 2, 3, 4, 5], 2), [1, 2, 3, 5]), (([1], 1), []), (([1, 2], 1), [1])], prep=lambda a: (mk_list(a[0]), a[1]), post=list_arr)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Count the length L, then remove node L − n from the front.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Move fast n steps ahead of slow (both start at a dummy). Move both until fast is at the last node; slow.next is the node to delete.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Count the length L, then remove node L − n from the front.

```python
def remove_nth_from_end(head, n):
    length, cur = 0, head
    while cur:
        length, cur = length + 1, cur.next
    dummy = ListNode(0, head)
    cur = dummy
    for _ in range(length - n):
        cur = cur.next
    cur.next = cur.next.next
    return dummy.next
```

**Cost:** O(n) — two passes time, O(1) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Move fast n steps ahead of slow (both start at a dummy). Move both until fast is at the last node; slow.next is the node to delete.

```python
def remove_nth_from_end(head, n):
    dummy = ListNode(0, head)
    slow = fast = dummy
    for _ in range(n):
        fast = fast.next
    while fast.next:
        slow, fast = slow.next, fast.next
    slow.next = slow.next.next
    return dummy.next
```

**Cost:** O(n) — one pass time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Count the length L, then remove node L − n from the front. That costs O(n) — two passes time and O(1) space."

"I can do better: Move fast n steps ahead of slow (both start at a dummy). Move both until fast is at the last node; slow.next is the node to delete. That's O(n) — one pass time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `head = [1, 2, 3, 4, 5], n = 2`

1. fast is 2 node(s) ahead of slow
2. Move both by one
3. Move both by one
4. Move both by one
5. slow.next (4) is n-th from the end → skip it

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/linked-list/remove-nth-node-from-end-of-list)

<a id="q10"></a>
### Q10. Reorder List

🟡 Interview standard · *Middle + reverse + weave*

Reorder L0 → L1 → … → Ln into L0 → Ln → L1 → Ln−1 → … in place.

**Examples**

- `head = [1, 2, 3, 4]` → `[1, 4, 2, 3]`
- `head = [1, 2, 3, 4, 5]` → `[1, 5, 2, 4, 3]`

In [ ]:
def reorder_list(head):
    """
    Reorder L0 → L1 → … → Ln into L0 → Ln → L1 → Ln−1 → … in place.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(reorder_list, [(([1, 2, 3, 4],), [1, 4, 2, 3]), (([1, 2, 3, 4, 5],), [1, 5, 2, 4, 3])], prep=L1, post=list_arr, inplace=0)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Put the nodes in an array and relink them with two indices from both ends.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Find the middle, reverse the second half, then alternately take a node from each half.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Put the nodes in an array and relink them with two indices from both ends.

```python
def reorder_list(head):
    nodes = []
    while head:
        nodes.append(head)
        head = head.next
    i, j = 0, len(nodes) - 1
    while i < j:
        nodes[i].next = nodes[j]
        i += 1
        if i == j:
            break
        nodes[j].next = nodes[i]
        j -= 1
    nodes[i].next = None
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Find the middle, reverse the second half, then alternately take a node from each half.

```python
def reorder_list(head):
    slow = fast = head
    while fast.next and fast.next.next:
        slow, fast = slow.next, fast.next.next
    second, slow.next = slow.next, None
    prev = None
    while second:
        second.next, prev, second = prev, second, second.next
    second = prev
    first = head
    while second:
        n1, n2 = first.next, second.next
        first.next, second.next = second, n1
        first, second = n1, n2
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Put the nodes in an array and relink them with two indices from both ends. That costs O(n) time and O(n) space."

"I can do better: Find the middle, reverse the second half, then alternately take a node from each half. That's O(n) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `head = [1, 2, 3, 4]`

1. 1) Split at the middle
2. 2) Reverse the second half
3. 3) Weave one node from each half
4. 3) Weave one node from each half

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/linked-list/reorder-list)

<a id="q11"></a>
### Q11. Add Two Numbers

🟡 Interview standard · *Digit-by-digit with carry*

Two non-negative numbers are stored as lists of digits in reverse order. Return their sum as a list in the same format.

**Examples**

- `l1 = [2, 4, 3], l2 = [5, 6, 4]` → `[7, 0, 8]`
- `l1 = [0], l2 = [0]` → `[0]`
- `l1 = [9, 9, 9, 9, 9, 9, 9], l2 = [9, 9, 9, 9]` → `[8, 9, 9, 9, 0, 0, 0, 1]`

In [ ]:
def add_two_numbers(l1, l2):
    """
    Two non-negative numbers are stored as lists of digits in reverse order.
    Return their sum as a list in the same format.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(add_two_numbers, [(([2, 4, 3], [5, 6, 4]), [7, 0, 8]), (([0], [0]), [0]), (([9, 9, 9, 9, 9, 9, 9], [9, 9, 9, 9]), [8, 9, 9, 9, 0, 0, 0, 1])], prep=lambda a: (mk_list(a[0]), mk_list(a[1])), post=list_arr)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Convert both lists to integers, add, convert back. (Overflows in languages with fixed-size ints.)

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Add digit by digit like on paper, carrying into the next node.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Convert both lists to integers, add, convert back. (Overflows in languages with fixed-size ints.)

```python
def add_two_numbers(l1, l2):
    def to_int(node):
        n, place = 0, 1
        while node:
            n += node.val * place
            node, place = node.next, place * 10
        return n
    total = to_int(l1) + to_int(l2)
    dummy = cur = ListNode()
    for d in reversed(str(total)):
        cur.next = ListNode(int(d))
        cur = cur.next
    return dummy.next
```

**Cost:** O(m + n) time, O(m + n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Add digit by digit like on paper, carrying into the next node.

```python
def add_two_numbers(l1, l2):
    dummy = tail = ListNode()
    carry = 0
    while l1 or l2 or carry:
        total = (l1.val if l1 else 0) + (l2.val if l2 else 0) + carry
        carry, digit = divmod(total, 10)
        tail.next = ListNode(digit)
        tail = tail.next
        l1 = l1.next if l1 else None
        l2 = l2.next if l2 else None
    return dummy.next
```

**Cost:** O(max(m, n)) time, O(1) extra space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Convert both lists to integers, add, convert back. (Overflows in languages with fixed-size ints.) That costs O(m + n) time and O(m + n) space."

"I can do better: Add digit by digit like on paper, carrying into the next node. That's O(max(m, n)) time and O(1) extra space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `l1 = [2, 4, 3], l2 = [5, 6, 4]`

1. 7 → write 7, carry 0
2. 10 → write 0, carry 1
3. 8 → write 8, carry 0

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/linked-list/add-two-numbers)

<a id="q12"></a>
### Q12. LRU Cache

🟡 Interview standard · *Hash map + doubly linked list*

Design a cache of fixed `capacity` with O(1) `get` and `put`; when full, evict the least recently used key.

**Examples**

- `LRUCache(2) → put(1, 1) → put(2, 2) → get(1) → put(3, 3) → get(2) → put(4, 4) → get(1) → get(3) → get(4)` → `[None, None, 1, None, -1, None, -1, 3, 4]`

In [ ]:
class LRUCache:
    """
    Design a cache of fixed capacity with O(1) get and put; when full, evict the
    least recently used key.
    """

    def __init__(self, capacity):
        pass

    def put(self, key, value):
        pass

    def get(self, key):
        pass


# ▶ Run this cell to test your answer
check(LRUCache, [
    (((2,), [['put', (1, 1)], ['put', (2, 2)], ['get', (1,)], ['put', (3, 3)], ['get', (2,)], ['put', (4, 4)], ['get', (1,)], ['get', (3,)], ['get', (4,)]]), [None, None, 1, None, -1, None, -1, 3, 4]),
], design=True)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Keep a list of (key, value) in recency order; every get/put searches and moves items.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A dict maps key → node in a doubly linked list ordered by recency. get moves the node to the front; put inserts at the front and evicts the node before the tail sentinel.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Keep a list of (key, value) in recency order; every get/put searches and moves items.

```python
class LRUCache:
    def __init__(self, capacity):
        self.cap, self.items = capacity, []  # most recent last

    def get(self, key):
        for i, (k, v) in enumerate(self.items):
            if k == key:
                self.items.append(self.items.pop(i))
                return v
        return -1

    def put(self, key, value):
        for i, (k, _) in enumerate(self.items):
            if k == key:
                self.items.pop(i)
                break
        self.items.append((key, value))
        if len(self.items) > self.cap:
            self.items.pop(0)
```

**Cost:** O(n) per op time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** A dict maps key → node in a doubly linked list ordered by recency. get moves the node to the front; put inserts at the front and evicts the node before the tail sentinel.

```python
class LRUCache:
    class Node:
        def __init__(self, key=0, val=0):
            self.key, self.val, self.prev, self.next = key, val, None, None

    def __init__(self, capacity):
        self.cap, self.map = capacity, {}
        self.head, self.tail = self.Node(), self.Node()  # sentinels
        self.head.next, self.tail.prev = self.tail, self.head

    def _remove(self, node):
        node.prev.next, node.next.prev = node.next, node.prev

    def _add_front(self, node):
        node.prev, node.next = self.head, self.head.next
        self.head.next.prev = node
        self.head.next = node

    def _order(self):
        out, cur = [], self.head.next
        while cur is not self.tail:
            out.append(f"{cur.key}:{cur.val}")
            cur = cur.next
        return out

    def get(self, key):
        if key not in self.map:
            return -1
        node = self.map[key]
        self._remove(node)
        self._add_front(node)
        return node.val

    def put(self, key, value):
        if key in self.map:
            self._remove(self.map[key])
        node = self.Node(key, value)
        self.map[key] = node
        self._add_front(node)
        if len(self.map) > self.cap:
            lru = self.tail.prev
            self._remove(lru)
            del self.map[lru.key]
```

**Cost:** O(1) per op time, O(capacity) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Keep a list of (key, value) in recency order; every get/put searches and moves items. That costs O(n) per op time and O(n) space."

"I can do better: A dict maps key → node in a doubly linked list ordered by recency. get moves the node to the front; put inserts at the front and evicts the node before the tail sentinel. That's O(1) per op time and O(capacity) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `LRUCache(2) → put(1, 1) → put(2, 2) → get(1) → put(3, 3) → get(2) → put(4, 4) → get(1) → get(3) → get(4)`

1. put(1, 1) at the front
2. put(2, 2) at the front
3. get(1) → 1; move it to the front
4. Over capacity → evict least recent key 2
5. put(3, 3) at the front
6. get(2) → miss, -1
7. Over capacity → evict least recent key 1
8. put(4, 4) at the front
9. get(1) → miss, -1
10. get(3) → 3; move it to the front
11. get(4) → 4; move it to the front

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/linked-list/lru-cache)

<a id="q13"></a>
### Q13. Rotate List

🟡 Interview standard · *Close the ring, cut*

Rotate the list to the right by `k` places.

**Examples**

- `head = [1, 2, 3, 4, 5], k = 2` → `[4, 5, 1, 2, 3]`
- `head = [0, 1, 2], k = 4` → `[2, 0, 1]`

In [ ]:
def rotate_right(head, k):
    """
    Rotate the list to the right by k places.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(rotate_right, [(([1, 2, 3, 4, 5], 2), [4, 5, 1, 2, 3]), (([0, 1, 2], 4), [2, 0, 1])], prep=lambda a: (mk_list(a[0]), a[1]), post=list_arr)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Move the last node to the front, k times.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Find length n and the tail, link tail to head to make a ring, then cut after node n − k % n − 1.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Move the last node to the front, k times.

```python
def rotate_right(head, k):
    if not head or not head.next:
        return head
    for _ in range(k):
        prev, cur = None, head
        while cur.next:
            prev, cur = cur, cur.next
        prev.next, cur.next, head = None, head, cur
    return head
```

**Cost:** O(n · k) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Find length n and the tail, link tail to head to make a ring, then cut after node n − k % n − 1.

```python
def rotate_right(head, k):
    if not head:
        return head
    n, tail = 1, head
    while tail.next:
        n, tail = n + 1, tail.next
    k %= n
    if k == 0:
        return head
    tail.next = head
    new_tail = head
    for _ in range(n - k - 1):
        new_tail = new_tail.next
    new_head = new_tail.next
    new_tail.next = None
    return new_head
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Move the last node to the front, k times. That costs O(n · k) time and O(1) space."

"I can do better: Find length n and the tail, link tail to head to make a ring, then cut after node n − k % n − 1. That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/linked-list/rotate-list)

<a id="q14"></a>
### Q14. Odd Even Linked List

🟡 Interview standard · *Two chains*

Group the nodes at odd positions first, then the even positions, keeping relative order.

**Examples**

- `head = [1, 2, 3, 4, 5]` → `[1, 3, 5, 2, 4]`
- `head = [2, 1, 3, 5, 6, 4, 7]` → `[2, 3, 6, 7, 1, 5, 4]`

In [ ]:
def odd_even_list(head):
    """
    Group the nodes at odd positions first, then the even positions, keeping
    relative order.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(odd_even_list, [(([1, 2, 3, 4, 5],), [1, 3, 5, 2, 4]), (([2, 1, 3, 5, 6, 4, 7],), [2, 3, 6, 7, 1, 5, 4])], prep=L1, post=list_arr)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Collect values by position parity and rebuild.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Build the odd chain and even chain at the same time by skipping every other node, then attach the even chain after the odd tail.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Collect values by position parity and rebuild.

```python
def odd_even_list(head):
    vals, i = [[], []], 0
    while head:
        vals[i % 2].append(head.val)
        head, i = head.next, i + 1
    return mk_list(vals[0] + vals[1])
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Build the odd chain and even chain at the same time by skipping every other node, then attach the even chain after the odd tail.

```python
def odd_even_list(head):
    if not head:
        return head
    odd, even = head, head.next
    even_head = even
    while even and even.next:
        odd.next = even.next
        odd = odd.next
        even.next = odd.next
        even = even.next
    odd.next = even_head
    return head
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Collect values by position parity and rebuild. That costs O(n) time and O(n) space."

"I can do better: Build the odd chain and even chain at the same time by skipping every other node, then attach the even chain after the odd tail. That's O(n) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `head = [1, 2, 3, 4, 5, 6, 7]`

1. Extend both chains by one
2. Extend both chains by one
3. Extend both chains by one

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/linked-list/odd-even-linked-list)

<a id="q15"></a>
### Q15. Sort List

🟡 Interview standard · *Merge sort on a list*

Sort a linked list in O(n log n) time.

**Examples**

- `head = [4, 2, 1, 3]` → `[1, 2, 3, 4]`
- `head = [-1, 5, 3, 4, 0]` → `[-1, 0, 3, 4, 5]`
- `head = []` → `[]`

In [ ]:
def sort_list(head):
    """
    Sort a linked list in O(n log n) time.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(sort_list, [(([4, 2, 1, 3],), [1, 2, 3, 4]), (([-1, 5, 3, 4, 0],), [-1, 0, 3, 4, 5]), (([],), [])], prep=L1, post=list_arr)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Copy values out, sort, write them back.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Merge sort: split at the middle with fast/slow, sort each half recursively, merge the two sorted halves.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Copy values out, sort, write them back.

```python
def sort_list(head):
    vals = []
    cur = head
    while cur:
        vals.append(cur.val)
        cur = cur.next
    return mk_list(sorted(vals))
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Merge sort: split at the middle with fast/slow, sort each half recursively, merge the two sorted halves.

```python
def sort_list(head):
    if not head or not head.next:
        return head
    slow, fast = head, head.next
    while fast and fast.next:
        slow, fast = slow.next, fast.next.next
    right, slow.next = slow.next, None
    a, b = sort_list(head), sort_list(right)
    dummy = tail = ListNode()
    while a and b:
        if a.val <= b.val:
            tail.next, a = a, a.next
        else:
            tail.next, b = b, b.next
        tail = tail.next
    tail.next = a or b
    return dummy.next
```

**Cost:** O(n log n) time, O(log n) recursion space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Copy values out, sort, write them back. That costs O(n log n) time and O(n) space."

"I can do better: Merge sort: split at the middle with fast/slow, sort each half recursively, merge the two sorted halves. That's O(n log n) time and O(log n) recursion space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `head = [5, 2, 7, 1, 3]`

1. Split into two halves
2. Split into two halves
3. Split into two halves
4. Merge two sorted halves
5. Merge two sorted halves
6. Split into two halves
7. Merge two sorted halves
8. Merge two sorted halves

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/linked-list/sort-list)

<a id="q16"></a>
### Q16. Swap Nodes in Pairs

🟡 Interview standard · *Pointer rewiring*

Swap every two adjacent nodes (change the links, not the values).

**Examples**

- `head = [1, 2, 3, 4]` → `[2, 1, 4, 3]`
- `head = []` → `[]`
- `head = [1, 2, 3]` → `[2, 1, 3]`

In [ ]:
def swap_pairs(head):
    """
    Swap every two adjacent nodes (change the links, not the values).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(swap_pairs, [(([1, 2, 3, 4],), [2, 1, 4, 3]), (([],), []), (([1, 2, 3],), [2, 1, 3])], prep=L1, post=list_arr)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Swap values in place — not allowed by the problem — or rebuild from an array.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

With a dummy before the head, for each pair a → b: prev → b, b → a, a → rest. Then prev = a.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Swap values in place — not allowed by the problem — or rebuild from an array.

```python
def swap_pairs(head):
    vals = []
    while head:
        vals.append(head.val)
        head = head.next
    for i in range(0, len(vals) - 1, 2):
        vals[i], vals[i + 1] = vals[i + 1], vals[i]
    return mk_list(vals)
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** With a dummy before the head, for each pair a → b: prev → b, b → a, a → rest. Then prev = a.

```python
def swap_pairs(head):
    dummy = ListNode(0, head)
    prev = dummy
    while prev.next and prev.next.next:
        a, b = prev.next, prev.next.next
        prev.next, a.next, b.next = b, b.next, a
        prev = a
    return dummy.next
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Swap values in place — not allowed by the problem — or rebuild from an array. That costs O(n) time and O(n) space."

"I can do better: With a dummy before the head, for each pair a → b: prev → b, b → a, a → rest. Then prev = a. That's O(n) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `head = [1, 2, 3, 4, 5, 6]`

1. Swapped 1 and 2
2. Swapped 3 and 4
3. Swapped 5 and 6

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/linked-list/swap-nodes-in-pairs)

<a id="q17"></a>
### Q17. Reverse Linked List II

🟡 Interview standard · *Reverse a sub-range*

Reverse the nodes from position `left` to `right` (1-indexed) in one pass.

**Examples**

- `head = [1, 2, 3, 4, 5], left = 2, right = 4` → `[1, 4, 3, 2, 5]`
- `head = [5], left = 1, right = 1` → `[5]`

In [ ]:
def reverse_between(head, left, right):
    """
    Reverse the nodes from position left to right (1-indexed) in one pass.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(reverse_between, [(([1, 2, 3, 4, 5], 2, 4), [1, 4, 3, 2, 5]), (([5], 1, 1), [5])], prep=lambda a: (mk_list(a[0]), a[1], a[2]), post=list_arr)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Copy values, reverse the slice, rebuild.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Walk to the node before 'left'. Then repeatedly take the node after the current sub-list tail and move it to the front of the sub-list (head insertion), right − left times.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Copy values, reverse the slice, rebuild.

```python
def reverse_between(head, left, right):
    vals = []
    while head:
        vals.append(head.val)
        head = head.next
    vals[left - 1:right] = vals[left - 1:right][::-1]
    return mk_list(vals)
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Walk to the node before 'left'. Then repeatedly take the node after the current sub-list tail and move it to the front of the sub-list (head insertion), right − left times.

```python
def reverse_between(head, left, right):
    dummy = ListNode(0, head)
    before = dummy
    for _ in range(left - 1):
        before = before.next
    cur = before.next
    for _ in range(right - left):
        move = cur.next
        cur.next = move.next
        move.next = before.next
        before.next = move
    return dummy.next
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Copy values, reverse the slice, rebuild. That costs O(n) time and O(n) space."

"I can do better: Walk to the node before 'left'. Then repeatedly take the node after the current sub-list tail and move it to the front of the sub-list (head insertion), right − left times. That's O(n) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `head = [1, 2, 3, 4, 5, 6], left = 2, right = 5`

1. Move 3 to the front of the sub-list
2. Move 4 to the front of the sub-list
3. Move 5 to the front of the sub-list

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/linked-list/reverse-linked-list-ii)

<a id="q18"></a>
### Q18. Partition List

🟡 Interview standard · *Two dummy lists*

Put all nodes with value &lt; `x` before nodes ≥ x, keeping the original order within each part.

**Examples**

- `head = [1, 4, 3, 2, 5, 2], x = 3` → `[1, 2, 2, 4, 3, 5]`
- `head = [2, 1], x = 2` → `[1, 2]`

In [ ]:
def partition(head, x):
    """
    Put all nodes with value < x before nodes ≥ x, keeping the original order
    within each part.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(partition, [(([1, 4, 3, 2, 5, 2], 3), [1, 2, 2, 4, 3, 5]), (([2, 1], 2), [1, 2])], prep=lambda a: (mk_list(a[0]), a[1]), post=list_arr)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Collect the two groups of values and rebuild.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Append each node to a 'less' list or a 'rest' list (each with a dummy head), then join less → rest and cut the tail.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Collect the two groups of values and rebuild.

```python
def partition(head, x):
    lo, hi = [], []
    while head:
        (lo if head.val < x else hi).append(head.val)
        head = head.next
    return mk_list(lo + hi)
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Append each node to a 'less' list or a 'rest' list (each with a dummy head), then join less → rest and cut the tail.

```python
def partition(head, x):
    less = lt = ListNode()
    rest = rt = ListNode()
    while head:
        if head.val < x:
            lt.next, lt = head, head
        else:
            rt.next, rt = head, head
        nxt = head.next
        head.next = None
        head = nxt
    lt.next = rest.next
    return less.next
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Collect the two groups of values and rebuild. That costs O(n) time and O(n) space."

"I can do better: Append each node to a 'less' list or a 'rest' list (each with a dummy head), then join less → rest and cut the tail. That's O(n) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `head = [1, 4, 3, 2, 5, 2], x = 3`

1. 1 goes to less
2. 4 goes to rest
3. 3 goes to rest
4. 2 goes to less
5. 5 goes to rest
6. 2 goes to less

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/linked-list/partition-list)

<a id="q19"></a>
### Q19. Merge k Sorted Lists

🔴 Stretch · *Min-heap of heads*

Merge `k` sorted linked lists into one sorted list.

**Examples**

- `lists = [[1, 4, 5], [1, 3, 4], [2, 6]]` → `[1, 1, 2, 3, 4, 4, 5, 6]`
- `lists = []` → `[]`
- `lists = [[]]` → `[]`

In [ ]:
def merge_k_lists(lists):
    """
    Merge k sorted linked lists into one sorted list.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(merge_k_lists, [(([[1, 4, 5], [1, 3, 4], [2, 6]],), [1, 1, 2, 3, 4, 4, 5, 6]), (([],), []), (([[]],), [])], prep=lambda a: ([mk_list(x) for x in a[0]],), post=list_arr)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Merge the lists one by one into a growing result.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Push every list's head into a min-heap. Pop the smallest node, append it, push its next. The heap never holds more than k nodes.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Merge the lists one by one into a growing result.

```python
def merge_k_lists(lists):
    def merge(a, b):
        dummy = t = ListNode()
        while a and b:
            if a.val <= b.val:
                t.next, a = a, a.next
            else:
                t.next, b = b, b.next
            t = t.next
        t.next = a or b
        return dummy.next
    res = None
    for l in lists:
        res = merge(res, l)
    return res
```

**Cost:** O(k · N) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Push every list's head into a min-heap. Pop the smallest node, append it, push its next. The heap never holds more than k nodes.

```python
def merge_k_lists(lists):
    import heapq
    heap = [(node.val, i, node) for i, node in enumerate(lists) if node]
    heapq.heapify(heap)
    dummy = tail = ListNode()
    while heap:
        val, i, node = heapq.heappop(heap)
        tail.next = node
        tail = node
        if node.next:
            heapq.heappush(heap, (node.next.val, i, node.next))
    return dummy.next
```

**Cost:** O(N log k) time, O(k) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Merge the lists one by one into a growing result. That costs O(k · N) time and O(1) space."

"I can do better: Push every list's head into a min-heap. Pop the smallest node, append it, push its next. The heap never holds more than k nodes. That's O(N log k) time and O(k) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `lists = [[1, 4, 5], [1, 3, 4], [2, 6]]`

1. Pop smallest head 1 (from list 0)
2. Pop smallest head 1 (from list 1)
3. Pop smallest head 2 (from list 2)
4. Pop smallest head 3 (from list 1)
5. Pop smallest head 4 (from list 0)
6. Pop smallest head 4 (from list 1)
7. Pop smallest head 5 (from list 0)
8. Pop smallest head 6 (from list 2)

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/linked-list/merge-k-sorted-lists)

<a id="q20"></a>
### Q20. Reverse Nodes in k-Group

🔴 Stretch · *Reverse in chunks*

Reverse the nodes k at a time; a final group shorter than k stays as is.

**Examples**

- `head = [1, 2, 3, 4, 5], k = 2` → `[2, 1, 4, 3, 5]`
- `head = [1, 2, 3, 4, 5], k = 3` → `[3, 2, 1, 4, 5]`

In [ ]:
def reverse_k_group(head, k):
    """
    Reverse the nodes k at a time; a final group shorter than k stays as is.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(reverse_k_group, [(([1, 2, 3, 4, 5], 2), [2, 1, 4, 3, 5]), (([1, 2, 3, 4, 5], 3), [3, 2, 1, 4, 5])], prep=lambda a: (mk_list(a[0]), a[1]), post=list_arr)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Copy values to an array, reverse each full chunk, rebuild.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

For each group: check k nodes exist, reverse them in place, and connect the previous group's tail to the new group head.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Copy values to an array, reverse each full chunk, rebuild.

```python
def reverse_k_group(head, k):
    vals = []
    while head:
        vals.append(head.val)
        head = head.next
    for i in range(0, len(vals) - len(vals) % k, k):
        vals[i:i + k] = vals[i:i + k][::-1]
    return mk_list(vals)
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** For each group: check k nodes exist, reverse them in place, and connect the previous group's tail to the new group head.

```python
def reverse_k_group(head, k):
    dummy = ListNode(0, head)
    group_prev = dummy
    while True:
        kth = group_prev
        for _ in range(k):
            kth = kth.next
            if not kth:
                return dummy.next
        group_next = kth.next
        prev, cur = group_next, group_prev.next
        while cur is not group_next:
            cur.next, prev, cur = prev, cur, cur.next
        first = group_prev.next
        group_prev.next = kth
        group_prev = first
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Copy values to an array, reverse each full chunk, rebuild. That costs O(n) time and O(n) space."

"I can do better: For each group: check k nodes exist, reverse them in place, and connect the previous group's tail to the new group head. That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/linked-list/reverse-nodes-in-k-group)

---
# Part 4: Interview questions, out loud

Cover the answer, **say yours out loud**, then open it and compare. Aim for two or three clear sentences, then an example.

<details>
<summary><b>1. Array vs linked list — when would you choose a linked list?</b></summary>

When you insert/delete at the front or in the middle often and rarely need access by index, or when the size changes unpredictably. For most other cases arrays win because of O(1) indexing and cache locality.

</details>

<details>
<summary><b>2. How do you detect a cycle in a linked list?</b></summary>

Floyd's tortoise and hare: slow moves 1 step, fast moves 2. If they ever meet, there is a cycle; if fast reaches None, there isn't. O(n) time, O(1) space. (A visited set also works with O(n) space.)

</details>

<details>
<summary><b>3. How do you find where a cycle starts?</b></summary>

After slow and fast meet, move one pointer back to head and advance both one step at a time. They meet at the cycle's entrance. Reason: the distance from head to the entrance equals the distance from the meeting point to the entrance (mod the cycle length).

</details>

<details>
<summary><b>4. Why is deleting a node O(1) in a doubly linked list but O(n) in a singly linked list?</b></summary>

Deletion needs the previous node to reconnect pointers. A doubly linked node stores `prev` directly; a singly linked list must walk from the head to find it.

</details>

<details>
<summary><b>5. What is a dummy (sentinel) node?</b></summary>

A placeholder node before the real head. It guarantees every real node has a predecessor, so inserting or deleting at the head needs no special case. Return `dummy.next` at the end.

</details>

<details>
<summary><b>6. How would you reverse a linked list? Iterative vs recursive?</b></summary>

Iterative: keep prev, cur, nxt and flip `cur.next = prev` while walking — O(n) time, O(1) space. Recursive: reverse the rest, then set `head.next.next = head`, `head.next = None` — O(n) time, O(n) stack space.

</details>

<details>
<summary><b>7. Why does an LRU cache use a doubly linked list?</b></summary>

It needs to move a recently used item to the front and remove the least recently used item from the back, both in O(1). A doubly linked list allows O(1) removal of any node given a reference (from the hash map), which a singly linked list or array can't.

</details>


---
# Check yourself before moving on

You're ready for the next notebook when you can tick these **without looking**:

- [ ] I can explain: Nodes and pointers
- [ ] I can explain: Insert, delete and the dummy head
- [ ] I can explain: Reversing a list
- [ ] I can explain: Fast and slow pointers
- [ ] I can explain: Merging and splitting
- [ ] I can explain: Combining techniques

**Revision drill.** Come back in a few days, set a 25-minute timer per question, and solve these from a blank cell:

- [ ] [Q16](#q16) Swap Nodes in Pairs
- [ ] [Q17](#q17) Reverse Linked List II
- [ ] [Q18](#q18) Partition List
- [ ] [Q19](#q19) Merge k Sorted Lists
- [ ] [Q20](#q20) Reverse Nodes in k-Group

Spaced repetition beats re-reading: redoing a question after you've partly forgotten it is what makes it stick.